## Import

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import duckdb
import pandas as pd

from sindex.metrics.citations import (
    merge_citations_dicts, 
    merge_citations_from_files, 
    merge_citations_from_files_fast,
    combine_citations,
)
from sindex.metrics.mentions import combine_mentions
from sindex.metrics.fairscores import merge_doi_fair_scores_ndjson_files, extrapolate_emdb_fair_scores
from sindex.utils.files import combine_ndjson_files, merge_ndjson_files_in_folder
from sindex.metrics.topics import enhance_topics
from sindex.metrics.batch_jobs import (
    batch_process_metadata_from_slim, 
    create_metadata_table,
    create_citations_table,
    create_mentions_table,
    create_fair_scores_table,
    create_topics_table,
    create_dataset_metrics_table,
    calculate_normalization_factors_topics,
    calculate_normalization_factors_subfields,
    create_dataset_index_table,
    create_creators_table,
    create_s_index_identifier_table,
    create_s_index_name_table,
    create_s_index_name_affiliation_table
)

## Citations

### Deduplicate citations from different sources

#### DataCite (DOIs)

In [24]:
mdc_citations = r"D:\pipeline-data\citations\mdc\mdc_citations.ndjson"
oa_citations = r"D:\pipeline-data\citations\openalex\oa_citations.ndjson"
dc_citations = r"D:\pipeline-data\citations\datacite\dc_citations.ndjson"
citation_files = [mdc_citations, oa_citations, dc_citations]
output_file_doi = r"D:\pipeline-data\citations\doi_citations.ndjson"

In [25]:
merge_citations_from_files_fast(citation_files, output_file_doi)

Starting merge of 3 valid files...
Finished processing 8,861,682 records. Unique: 7,654,146
Writing to D:\pipeline-data\citations\doi_citations.ndjson...
Done!


#### EMDB

In [17]:
mdc_citations = r"I:\pipeline-data\citations\mdc\mdc_citations_emdb.ndjson"
citation_files = [mdc_citations]
output_file_emdb = r"I:\pipeline-data\citations\emdb_citations.ndjson"

In [18]:
merge_citations_from_files_fast(citation_files, output_file_emdb)

Starting merge of 1 valid files...
Finished processing 15,134 records. Unique: 15,134
Writing to I:\pipeline-data\citations\emdb_citations.ndjson...
Done!


### Combine and add placeholder dates when citation date missing

In [26]:
doi = r"D:\pipeline-data\citations\doi_citations.ndjson"
emdb = r"D:\pipeline-data\citations\emdb_citations.ndjson"
file_list = [doi, emdb]
output_path = r"D:\pipeline-data\citations\citations.ndjson"

In [27]:
combine_ndjson_files(file_list, output_path)

Lines processed: 7,660,000
Finished! Total entries saved: 7,669,280


## Mentions

### Combine and add placeholder dates when mention date missing

In [12]:
mock = r"D:\pipeline-data\mentions\mentions_github_mock.ndjson"
file_list = [mock]
output_path = r"D:\pipeline-data\mentions\mentions.ndjson"

In [13]:
combine_mentions(file_list, output_path)

Lines processed: 4,150,000
Finished! Total entries saved: 4,156,510


## FAIR scores

### Merge DOI FAIR score into one file (rename doi field as dataset_id)

In [40]:
fair_scores_directory = r"D:\pipeline-data\fair_scores\fair_scores_doi_files"
doi_fair_scores_path = r"D:\pipeline-data\fair_scores\doi_fair_scores.ndjson"

In [41]:
merge_doi_fair_scores_ndjson_files(fair_scores_directory, doi_fair_scores_path)

Found 4901 files. Starting merge with orjson...
Done! Total lines in 'D:\pipeline-data\fair_scores\doi_fair_scores.ndjson': 49,009,521    


### Extrapolate EMDB fair scores (all the same on the first 10k calculated with F-UJI)

In [42]:
emdb_file_path = r"D:\pipeline-data\records\slim-records\emdb-slim-records\emdb-records-slim.ndjson"
partial_score_file_path = r"D:\pipeline-data\fair_scores\fair_scores_emdb_partial\fair_scores_emdb_partial.ndjson"
emdb_fair_scores_path = r"D:\pipeline-data\fair_scores\emdb_fair_scores.ndjson"

In [30]:
extrapolate_emdb_fair_scores(emdb_file_path, partial_score_file_path, emdb_fair_scores_path)

Loading scores from D:\pipeline-data\fair_scores\fair_scores_emdb_partial\fair_scores_emdb_partial.ndjson...
Processing D:\pipeline-data\records\slim-records\emdb-slim-records\emdb-records-slim.ndjson...
----------------------------------------
STATISTICS (orjson)
----------------------------------------
Total records in EMDB file:   51645
Total records written:        51645
  - Found existing scores:    18132
  - Extrapolated scores:      33513
  - Skipped (no ID):          0
----------------------------------------
SUCCESS: Input count matches output count.


### Merge all FAIR score into one file

In [43]:
fair_scores_path = r"D:\pipeline-data\fair_scores\fair_scores.ndjson"

In [44]:
combine_ndjson_files([doi_fair_scores_path, emdb_fair_scores_path], fair_scores_path)

Lines processed: 49,000,000
Finished! Total entries saved: 49,061,166


## Topics

### Enhance fair scores with subfiled, field, and domain

In [9]:
input_topics = r"D:\pipeline-data\topics\topics.ndjson"
mapping_file = r"D:\pipeline-data\external\openalex-topics\openalex_topic_mapping_table.csv"
output_topics = r"D:\pipeline-data\topics\topics_enhanced.ndjson"

In [10]:
enhance_topics(input_topics, mapping_file, output_topics)

--- Starting Line-by-Line Enhancement ---
Loading mapping CSV...
Mapping loaded. 4,516 topics indexed.
Sample Key: 'T10001'
Processing lines...
Lines processed: 15,300,000 | Matches: 15,300,000

--- Done! ---
Total Lines: 15,324,819
Total Matches: 15,324,819
Saved to: D:\pipeline-data\topics\topics_enhanced.ndjson


## Dataset report

### Dataset metadata

#### Create metadata njson files so easier to load in table

In [3]:
slim_folder = r"D:\pipeline-data\records\slim-records"
dst_folder = r"D:\pipeline-data\dataset_index\metadata-records"

In [4]:
batch_process_metadata_from_slim(slim_folder, dst_folder)

Processing 772 files using 32 cores...
Input: D:\pipeline-data\records\slim-records
Output: D:\pipeline-data\dataset_index\metadata-records
[772/772] files completed
Done. files=772 kept=49,061,167 bad=0 time=999.3s rate≈49,097/rec-per-sec


{'files_seen': 772,
 'records_read': 49061167,
 'records_kept': 49061167,
 'records_bad_json': 0,
 'output_dir': 'D:\\pipeline-data\\dataset_index\\metadata-records',
 'elapsed_sec': 999.26,
 'rate_rec_per_sec': 49097}

#### Load metadata in duckdb table

In [13]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"
metadata_folder = r"D:\pipeline-data\dataset_index\metadata-records"

In [14]:
create_metadata_table(dataset_reports_db, metadata_folder)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Metadata table created in 'D:\pipeline-data\dataset_index\dataset_reports.duckdb'. Total rows: 49061167

Sample rows


ModuleNotFoundError: No module named 'matplotlib'

In [19]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM metadata LIMIT 3").df())
con.close()

,dataset_id,pub_ts,pubyear,creators,title,source
0,10.4225/15/515b76f4dbf24,2011-01-01,2011,"[{""name"":""WESTWOOD, KAREN JILLIAN"",""name_type""...",Primary Production in the Sub-Antarctic and Po...,datacite
1,10.4225/15/515b7978e65a0,2010-01-01,2010,"[{""name"":""WESTWOOD, KAREN JILLIAN"",""name_type""...","Primary productivity, pulse amplitude modulate...",datacite
2,10.1594/pangaea.808335,2012-01-01,2012,"[{""name"":""Glas, Martin S""},{""name"":""Langer, Ge...",(Figure 4) pH and Ca**2+ dynamics of an adult ...,datacite


In [20]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM metadata WHERE source = 'emdb' LIMIT 3").df())
con.close()

,dataset_id,pub_ts,pubyear,creators,title,source
0,EMD-48024,2024-11-21,2024,"[{""name"":""Markert J"",""name_type"":""Personal""},{...",map beta Paf1C,emdb
1,EMD-70460,2025-04-30,2025,"[{""name"":""Wing CE"",""name_type"":""Personal"",""ide...",Cryo-EM structure of human exportin-1 conjugat...,emdb
2,EMD-60222,2024-05-17,2024,"[{""name"":""Chen MY"",""name_type"":""Personal""},{""n...",Structure of Polycystin-1/Polycystin-2 complex...,emdb


### Citations, Mentions, FAIR scores, and Topics

In [13]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"
citations_file =  r"D:\pipeline-data\citations\citations.ndjson"
mentions_file =  r"D:\pipeline-data\mentions\mentions.ndjson"
fair_scores_file =  r"D:\pipeline-data\fair_scores\fair_scores.ndjson"
topics_file =  r"D:\pipeline-data\topics\topics_enhanced.ndjson"

#### Load citations to duckdb

In [27]:
create_citations_table(dataset_reports_db, citations_file)

Loading Citations from: D:\pipeline-data\citations\citations.ndjson


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Citations table created. Rows: 7669280
        dataset_id     cit_ts  citation_year  citation_weight  \
0  10.5517/cct09bf 2010-05-25           2010             1.11   
1  10.5517/ccst4mb 2010-05-25           2010             1.11   
2  10.5517/ccv7428 2010-05-25           2010             1.11   
3  10.5517/ccv7439 2010-05-25           2010             1.11   
4  10.5517/ccv744b 2010-05-26           2010             1.11   

               source  
0  ["datacite","mdc"]  
1  ["datacite","mdc"]  
2  ["datacite","mdc"]  
3  ["datacite","mdc"]  
4             ["mdc"]  


#### Load mentions to duckdb

In [32]:
create_mentions_table(dataset_reports_db, mentions_file)

Loading Mentions from: D:\pipeline-data\mentions\mentions.ndjson


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Mentions table created. Rows: 4,156,510
        dataset_id     men_ts  mention_year  mention_weight   source
0  10.5517/cct09bf 2010-05-26          2010            1.11  ["mdc"]
1  10.5517/ccst4mb 2010-05-26          2010            1.11  ["mdc"]
2  10.5517/ccv7428 2010-05-26          2010            1.11  ["mdc"]
3  10.5517/ccv7439 2010-05-26          2010            1.11  ["mdc"]
4  10.5517/ccv744b 2010-05-26          2010            1.11  ["mdc"]


#### Load FAIR scores to duckdb

In [45]:
create_fair_scores_table(dataset_reports_db, fair_scores_file)

Loading FAIR Scores from: D:\pipeline-data\fair_scores\fair_scores.ndjson


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FAIR Scores table created. Rows: 49,061,166
        dataset_id  score
0  10.5284/1000389  30.77
1  10.5284/1000140  30.77
2  10.5284/1000146  50.00
3  10.5284/1000144  30.77
4  10.5284/1000181  30.77


#### Load topics to duckdb

In [14]:
create_topics_table(dataset_reports_db, topics_file)

Loading Topics from: D:\pipeline-data\topics\topics_enhanced.ndjson


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Topics table created. Rows: 15,324,819
                    dataset_id topic_id  \
0  10.57451/lhd.a.ha3.166772.1   T10765   
1  10.57451/lhd.a.ha3.167696.1   T10093   
2  10.57451/lhd.a.ha3.167781.1   T10252   
3  10.57451/lhd.a.ha3.168157.1   T14339   
4  10.57451/lhd.a.ha3.168181.1   T11259   

                                          topic_name     score    source  \
0                Marine Biology and Ecology Research  0.791819  openalex   
1                   Nuclear physics research studies  0.116844  openalex   
2        Microbial Natural Products and Biosynthesis  0.354062  openalex   
3             Image Processing and 3D Reconstruction  0.050859  openalex   
4  Agriculture Sustainability and Environmental I...  0.242771  openalex   

  subfield_id                            subfield_name field_id  \
0        1910                             Oceanography       19   
1        3106          Nuclear and High Energy Physics       31   
2        2736                             Ph

## Normalization factors

In [3]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"

### Create table with all the required metrics for each dataset

In [13]:
create_dataset_metrics_table(dataset_reports_db)

Creating dataset_metrics table (topics, creators, FAIR, and 3-year metrics)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dataset_metrics table created. Total Datasets: 49,061,167

Sample rows (verifying creators column):
                dataset_id                                           creators  \
0  10.5281/zenodo.10347789  [{"name":"WirtualneMuzeaMalopolski","name_type...   
1  10.5281/zenodo.10347790  [{"name":"WirtualneMuzeaMalopolski","name_type...   
2  10.5281/zenodo.10347793  [{"name":"hsjapan3dcg","name_type":"Personal",...   
3  10.5281/zenodo.10347794  [{"name":"hsjapan3dcg","name_type":"Personal",...   
4  10.5281/zenodo.10347795  [{"name":"www.noe-3d.at","name_type":"Personal...   

                                  topic_name  cit_3yr  
0  Consumer Packaging Perceptions and Trends        0  
1  Consumer Packaging Perceptions and Trends        0  
2      Ginkgo biloba and Cashew Applications        0  
3      Ginkgo biloba and Cashew Applications        0  
4       Medical and Health Sciences Research        0  


In [4]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM dataset_metrics LIMIT 3").df())
con.close()

,dataset_id,pubyear,creators,dataset_source,topic_id,topic_name,topic_score,subfield_id,subfield_name,field_id,...,domain_name,fair_score,total_citations,total_cit_weight,cit_3yr,cit_weight_3yr,total_mentions,total_men_weight,men_3yr,men_weight_3yr
0,10.5281/zenodo.10347789,2019,"[{""name"":""WirtualneMuzeaMalopolski"",""name_type...",datacite,T14055,Consumer Packaging Perceptions and Trends,0.8692,1406,Marketing,14,...,Social Sciences,13.46,0,0.0,0,0.0,0,0.0,0,0.0
1,10.5281/zenodo.10347790,2019,"[{""name"":""WirtualneMuzeaMalopolski"",""name_type...",datacite,T14055,Consumer Packaging Perceptions and Trends,0.8692,1406,Marketing,14,...,Social Sciences,51.92,0,0.0,0,0.0,0,0.0,0,0.0
2,10.5281/zenodo.10347793,2022,"[{""name"":""hsjapan3dcg"",""name_type"":""Personal"",...",datacite,T12846,Ginkgo biloba and Cashew Applications,0.5917,2707,Complementary and alternative medicine,27,...,Health Sciences,51.92,0,0.0,0,0.0,0,0.0,0,0.0


In [5]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM dataset_metrics ORDER BY pubyear DESC LIMIT 5").df())
con.close()

,dataset_id,pubyear,creators,dataset_source,topic_id,topic_name,topic_score,subfield_id,subfield_name,field_id,...,domain_name,fair_score,total_citations,total_cit_weight,cit_3yr,cit_weight_3yr,total_mentions,total_men_weight,men_3yr,men_weight_3yr
0,10.15151/esrf-es-1059522434,2026,"[{""name"":""LUKIC, Bratislav"",""name_type"":""Perso...",datacite,T10451,Mycorrhizal Fungi and Plant Interactions,0.491003,1110,Plant Science,11,...,Life Sciences,15.38,0,0.0,0,0.0,0,0.0,0,0.0
1,10.15151/esrf-es-1031681706,2026,"[{""name"":""RUZICKA, Barbara"",""name_type"":""Perso...",datacite,T10451,Mycorrhizal Fungi and Plant Interactions,0.561539,1110,Plant Science,11,...,Life Sciences,15.38,0,0.0,0,0.0,0,0.0,0,0.0
2,10.15151/esrf-es-1030814352,2026,"[{""name"":""Aiyarin KITTILUKKANA""},{""name"":""CARM...",datacite,T10451,Mycorrhizal Fungi and Plant Interactions,0.557453,1110,Plant Science,11,...,Life Sciences,15.38,0,0.0,0,0.0,0,0.0,0,0.0
3,10.15151/esrf-es-1030391080,2026,"[{""name"":""SERRANO, Aida"",""name_type"":""Personal...",datacite,T10451,Mycorrhizal Fungi and Plant Interactions,0.569982,1110,Plant Science,11,...,Life Sciences,15.38,0,0.0,0,0.0,0,0.0,0,0.0
4,10.15151/esrf-es-1030395172,2026,"[{""name"":""SARMA, Bidyut Bikash"",""name_type"":""P...",datacite,T10451,Mycorrhizal Fungi and Plant Interactions,0.495650,1110,Plant Science,11,...,Life Sciences,15.38,0,0.0,0,0.0,0,0.0,0,0.0


In [7]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT dataset_id, creators FROM dataset_metrics LIMIT 3").df())
con.close()

,dataset_id,creators
0,10.5281/zenodo.10347789,"[{""name"":""WirtualneMuzeaMalopolski"",""name_type..."
1,10.5281/zenodo.10347790,"[{""name"":""WirtualneMuzeaMalopolski"",""name_type..."
2,10.5281/zenodo.10347793,"[{""name"":""hsjapan3dcg"",""name_type"":""Personal"",..."


In [81]:
con = duckdb.connect(dataset_reports_db)
schema_df = con.execute("DESCRIBE dataset_metrics").df()
display(schema_df)
con.close()

,column_name,column_type,null,key,default,extra
0,dataset_id,VARCHAR,YES,None,None,None
1,pubyear,INTEGER,YES,None,None,None
2,creators,JSON,YES,None,None,None
3,dataset_source,VARCHAR,YES,None,None,None
4,topic_id,VARCHAR,YES,None,None,None
5,topic_name,VARCHAR,YES,None,None,None
6,topic_score,DOUBLE,YES,None,None,None
7,subfield_id,VARCHAR,YES,None,None,None
8,subfield_name,VARCHAR,YES,None,None,None
9,field_id,VARCHAR,YES,None,None,None


### Create normalization factors by topics table

In [26]:
calculate_normalization_factors_topics(dataset_reports_db)

Creating normalization_factors_topics table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Generating Global Benchmark...
   Generating benchmarks for 4516 topics...
Generating rolling medians for target years...
Saving 343292 benchmark rows...
normalization_factors_topics table created.

--- Sample view (Lifetime Topic Metrics) ---
  topic_id                               topic_name  median_cit_weight_3yr  \
0   T10001      Geological and Geochemical Analysis                    0.0   
1   T10002        Advanced Chemical Physics Studies                    0.0   
2   T10003      Innovation and Knowledge Management                    0.0   
3   T10004        Soil Carbon and Nitrogen Dynamics                    0.0   
4   T10005  Ecology and Vegetation Dynamics Studies                    0.0   

   n_cit  
0   9683  
1   2596  
2    861  
3   2770  
4   2781  


In [27]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM normalization_factors_topics WHERE n_cit > 20 ORDER BY median_cit_weight_3yr DESC LIMIT 5").df())
con.close()

,topic_id,topic_name,pubyear,median_fair_score_3yr,median_cit_weight_3yr,median_men_weight_3yr,n_fair,n_cit,n_men
0,T10043,Substance Abuse Treatment and Outcomes,2000,13.46,1.45,0.00,28,25,25
1,T11719,Data Quality and Management,2018,15.38,1.27,1.27,599,468,468
2,T14187,Varied Academic Research Topics,2018,15.38,1.27,1.27,35,33,33
3,T11719,Data Quality and Management,2019,15.38,1.27,1.27,668,458,458
4,T14187,Varied Academic Research Topics,2019,15.38,1.27,1.27,35,34,34


In [92]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM normalization_factors_topics WHERE topic_id = 'T12174' AND pubyear = 1991 LIMIT 20").df())
con.close()

,topic_id,topic_name,pubyear,median_fair_score_3yr,median_cit_weight_3yr,median_men_weight_3yr,n_fair,n_cit,n_men
0,T12174,Hospital Admissions and Outcomes,1991,13.46,5.305,0.0,20,16,16


### Create normalization factors by subfield table

In [23]:
calculate_normalization_factors_subfields(dataset_reports_db)

Calculating Normalization Factors by Subfield (Guaranteed Benchmarks)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Generating Global Benchmark...
   Generating benchmarks for 252 subfields...
Generating rolling medians for target years...
Saving 19228 benchmark rows...
Normalization subfields table created.

--- Sample view (Lifetime Subfield Metrics) ---
  subfield_id                                 subfield_name  \
0        1100  General Agricultural and Biological Sciences   
1        1102                     Agronomy and Crop Science   
2        1103                    Animal Science and Zoology   
3        1104                               Aquatic Science   
4        1105  Ecology, Evolution, Behavior and Systematics   

   median_cit_weight_3yr   n_cit  
0                    0.0   24601  
1                    0.0   16848  
2                    0.0   11905  
3                    0.0   11362  
4                    0.0  126347  


In [25]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM normalization_factors_subfields WHERE n_cit > 20 ORDER BY median_cit_weight_3yr DESC LIMIT 5").df())
con.close()

,subfield_id,subfield_name,pubyear,median_fair_score_3yr,median_cit_weight_3yr,median_men_weight_3yr,n_fair,n_cit,n_men
0,2505,Materials Chemistry,1998,13.46,1.06,1.06,4698,2446,2446
1,3106,Nuclear and High Energy Physics,2013,13.46,1.05,0.00,279,161,161
2,2505,Materials Chemistry,1999,13.46,1.04,1.04,7553,4343,4343
3,2505,Materials Chemistry,2017,13.46,1.03,1.04,133897,87675,87675
4,2505,Materials Chemistry,2018,13.46,1.01,1.00,142864,93009,93009


In [30]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM normalization_factors_subfields WHERE subfield_id = 2505 ORDER BY pubyear DESC LIMIT 20").df())
con.close()

,subfield_id,subfield_name,pubyear,median_fair_score_3yr,median_cit_weight_3yr,median_men_weight_3yr,n_fair,n_cit,n_men
0,2505,Materials Chemistry,2027,13.46,0.00,0.00,19530,19526,19526
1,2505,Materials Chemistry,2026,13.46,0.00,0.00,150666,142443,142443
2,2505,Materials Chemistry,2025,13.46,0.00,0.00,195197,183894,183894
3,2505,Materials Chemistry,2024,13.46,0.00,0.00,245682,114542,114542
4,2505,Materials Chemistry,2023,13.46,0.00,0.00,187146,134392,134392
5,2505,Materials Chemistry,2022,13.46,1.00,0.00,190444,128656,128656
6,2505,Materials Chemistry,2021,13.46,0.00,0.00,201710,129106,129106
7,2505,Materials Chemistry,2020,13.46,1.00,0.00,178961,122909,122909
8,2505,Materials Chemistry,2019,13.46,1.01,0.00,169131,96077,96077
9,2505,Materials Chemistry,2018,13.46,1.01,1.00,142864,93009,93009


In [89]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM normalization_factors_subfields WHERE subfield_id = 1802 ORDER BY pubyear DESC LIMIT 20").df())
con.close()

BinderException: Binder Error: Referenced column "topic_id" not found in FROM clause!
Candidate bindings: "median_cit_weight_3yr", "median_fair_score_3yr", "subfield_id"

LINE 1: SELECT * FROM normalization_factors_subfields WHERE topic_id = T12174 ORDER BY pubyear DESC LIMIT 20
                                                            ^

### Create adjusted normalization factors table (0 values replaced with 0.3 for median FAIR score and 1 for median citations and mentions

## Dataset Index

In [49]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"

In [50]:
create_dataset_index_table(dataset_reports_db)

Initializing dataset_index creation


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! Table 'dataset_index' created.
Total Rows: 49,061,167
Execution Time: 912.32 seconds


In [88]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT pubyear, topic_id, subfield_id, fair_score, t_norm_fair_final, total_cit_weight, t_norm_cit_final, total_men_weight, t_norm_men_final, dataset_index_topic, FROM dataset_index WHERE t_norm_cit_final > 1 ORDER BY total_cit_weight DESC LIMIT 5").df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pubyear,topic_id,subfield_id,fair_score,t_norm_fair_final,total_cit_weight,t_norm_cit_final,total_men_weight,t_norm_men_final,dataset_index_topic
0,1991,T12174,2711,69.23,20.00,2554.67,5.305,2554.67,1.00,1013.230136
1,1984,T10443,3320,13.46,20.00,821.70,17.790,821.70,17.79,31.016913
2,1979,T13122,1203,13.46,73.08,573.28,1.110,385.18,1.11,287.887220
3,2009,T14445,2205,13.46,20.00,564.32,11.890,518.94,11.93,30.544492
4,2002,T12795,1110,13.46,34.62,561.03,1.030,402.78,1.03,312.042219


In [52]:
con = duckdb.connect(dataset_reports_db)
schema_df = con.execute("DESCRIBE dataset_index").df()
display(schema_df)
con.close()

,column_name,column_type,null,key,default,extra
0,dataset_id,VARCHAR,YES,None,None,None
1,pubyear,INTEGER,YES,None,None,None
2,creators,JSON,YES,None,None,None
3,dataset_source,VARCHAR,YES,None,None,None
4,topic_id,VARCHAR,YES,None,None,None
5,topic_name,VARCHAR,YES,None,None,None
6,topic_score,DOUBLE,YES,None,None,None
7,subfield_id,VARCHAR,YES,None,None,None
8,subfield_name,VARCHAR,YES,None,None,None
9,field_id,VARCHAR,YES,None,None,None


In [53]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT dataset_id, dataset_source, creators, dataset_index_topic, dataset_index_subfield FROM dataset_index LIMIT 5").df())
con.close()

,dataset_id,dataset_source,creators,dataset_index_topic,dataset_index_subfield
0,10.15156/bio/sh3353015.08fu,datacite,"[{""name"":""Kõljalg, Urmas"",""name_type"":""Persona...",0.512833,0.512833
1,10.25574/61937,datacite,"[{""name"":""CXC-DS"",""name_type"":""Organizational""...",0.140303,0.121169
2,10.15156/bio/sh3353018.08fu,datacite,"[{""name"":""Kõljalg, Urmas"",""name_type"":""Persona...",0.512833,0.512833
3,10.25574/61938,datacite,"[{""name"":""CXC-DS"",""name_type"":""Organizational""...",0.121169,0.121169
4,10.15156/bio/sh3353025.08fu,datacite,"[{""name"":""Kõljalg, Urmas"",""name_type"":""Persona...",0.512833,0.512833


In [95]:
con = duckdb.connect(dataset_reports_db)
query = """
        SELECT 
            dataset_id, 
            dataset_source, 
            creators,
            dataset_index_topic,
            dataset_index_subfield
        FROM dataset_index 
        WHERE 
            creators LIKE '%0000-0003-0770-6869%'
        LIMIT 10
        """
display(con.execute(query).df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset_id,dataset_source,creators,dataset_index_topic,dataset_index_subfield
0,10.5281/zenodo.4713727,datacite,"[{""name"":""Preciado-Velasco, Jorge"",""identifier...",1.153833,1.153833
1,10.5281/zenodo.4713726,datacite,"[{""name"":""Preciado-Velasco, Jorge"",""identifier...",1.153833,1.153833
2,10.5281/zenodo.4779074,datacite,"[{""name"":""Preciado-Velasco, Jorge E."",""identif...",1.346167,1.346167
3,10.5281/zenodo.4779073,datacite,"[{""name"":""Preciado-Velasco, Jorge E."",""identif...",1.346167,1.346167


In [108]:
con = duckdb.connect(dataset_reports_db)
query = """
        SELECT *
        FROM dataset_index 
        WHERE 
            dataset_id = '10.5281/zenodo.4779073'
        LIMIT 10
        """
df4 = con.execute(query).df()
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [116]:
for df in [df1, df2, df3, df4]:
    display(df[["pubyear", "fair_score", "t_norm_fair_final", "total_cit_weight", "t_norm_cit_final"]])

,pubyear,fair_score,t_norm_fair_final,total_cit_weight,t_norm_cit_final
0,2021,69.23,20.0,0.0,1.0


,pubyear,fair_score,t_norm_fair_final,total_cit_weight,t_norm_cit_final
0,2021,69.23,20.0,0.0,1.0


,pubyear,fair_score,t_norm_fair_final,total_cit_weight,t_norm_cit_final
0,2021,80.77,20.0,0.0,1.0


,pubyear,fair_score,t_norm_fair_final,total_cit_weight,t_norm_cit_final
0,2021,80.77,20.0,0.0,1.0


In [106]:
s

'[{"name":"Preciado-Velasco, Jorge","identifiers":["https://orcid.org/0000-0003-4543-2301"],"affiliations":["CICESE"]},{"name":"Gonzalez-Franco, Joan","identifiers":["https://orcid.org/0000-0003-0770-6869"],"affiliations":["CUJAE"]}]'

## S-index

In [4]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"

### Create a creators_table first exploding the dataset_index table on creators first

In [74]:
create_creators_table(dataset_reports_db)

FULL RUN: Creating 'creators_table' (Exploded with Context & Raw Metrics)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--------------------------------------------------
Success! 'creators_table' created.
Total Exploded Rows: 216,688,512
Execution Time: 13301.18 seconds
--------------------------------------------------

Preview of Identifier Normalization:
                 creator_name   primary_identifier
0    Tschonghongei, Nelson C.  0009-0007-9932-8135
1    Tschonghongei, Nelson C.  0009-0007-9932-8135
2  Ramos-Ordoñez, María Felix  0000-0002-9470-6375
3  Ramos-Ordoñez, María Felix  0000-0002-9470-6375
4  Ramos-Ordoñez, María Felix  0000-0002-9470-6375


In [75]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM creators_table ORDER BY total_cit_weight DESC LIMIT 5").df())
con.close()

,dataset_id,pubyear,topic_id,topic_name,subfield_id,subfield_name,creator_name,name_type,primary_identifier,affiliations,dataset_index_topic,dataset_index_subfield,total_cit_weight,total_men_weight,fair_score,total_citations,total_mentions
0,10.5519/0002965,2014,T11986,Scientific Computing and Data Management,1802,Information Systems and Management,Natural History Museum,Organizational,https://ror.org/039zvsn29,NaN,11885.491333,11885.491333,35541.60,111.22,73.08,21537,66
1,10.15468/ab3s5x,2025,T14423,Military Technology and Strategies,2202,Aerospace Engineering,iNaturalist contributors,NaN,NaN,"[""iNaturalist""]",10388.819000,10388.819000,30950.69,212.69,61.54,30948,210
2,10.15468/hnhrg3,2025,T10895,Species Distribution and Climate Change,2302,Ecological Modeling,Informatics and Data Science Center-Digital St...,NaN,NaN,"[""National Museum of Natural History, Smithson...",15709.769667,15709.769667,28524.52,18601.52,65.38,28524,18601
3,10.15468/hnhrg3,2025,T10895,Species Distribution and Climate Change,2302,Ecological Modeling,"Orrell, Thomas",Personal,0000-0003-1038-3028,"[""National Museum of Natural History, Smithson...",15709.769667,15709.769667,28524.52,18601.52,65.38,28524,18601
4,10.15468/ib5ypt,2025,T12568,Plant Taxonomy and Phylogenetics,1105,"Ecology, Evolution, Behavior and Systematics","Creuwels, Jeroen",Personal,0000-0001-6131-7026,"[""Naturalis Biodiversity Center""]",13300.269667,13300.269667,28171.27,11726.27,65.38,28171,11726


In [76]:
con = duckdb.connect(dataset_reports_db)
schema_df = con.execute("DESCRIBE creators_table").df()
display(schema_df)
con.close()

,column_name,column_type,null,key,default,extra
0,dataset_id,VARCHAR,YES,None,None,None
1,pubyear,INTEGER,YES,None,None,None
2,topic_id,VARCHAR,YES,None,None,None
3,topic_name,VARCHAR,YES,None,None,None
4,subfield_id,VARCHAR,YES,None,None,None
5,subfield_name,VARCHAR,YES,None,None,None
6,creator_name,VARCHAR,YES,None,None,None
7,name_type,VARCHAR,YES,None,None,None
8,primary_identifier,VARCHAR,YES,None,None,None
9,affiliations,JSON,YES,None,None,None


In [77]:
con = duckdb.connect(dataset_reports_db)
query = """
        SELECT 
            creator_name, 
            primary_identifier, 
            affiliations,
            dataset_index_topic
        FROM creators_table 
        WHERE regexp_matches(primary_identifier, '^[0-9]{4}-[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]$')
        LIMIT 10
        """
display(con.execute(query).df())
con.close()

,creator_name,primary_identifier,affiliations,dataset_index_topic
0,"Tschonghongei, Nelson C.",0009-0007-9932-8135,"[""University at Buffalo, State University of N...",0.224333
1,"Tschonghongei, Nelson C.",0009-0007-9932-8135,"[""University at Buffalo, State University of N...",0.224333
2,"Ramos-Ordoñez, María Felix",0000-0002-9470-6375,"[""Universidad Nacional Autónoma de México, Fac...",0.512833
3,"Ramos-Ordoñez, María Felix",0000-0002-9470-6375,"[""Universidad Nacional Autónoma de México, Fac...",0.512833
4,"Ramos-Ordoñez, María Felix",0000-0002-9470-6375,"[""Universidad Nacional Autónoma de México, Fac...",0.512833
5,"Villas Boas de Lima, Gabriel",0000-0001-7089-7421,"[""Universidade Federal do Pará""]",1.218000
6,"Han, Xiaoxiang",0000-0002-1946-2067,"[""Shanghai University""]",0.224333
7,"Han, Xiaoxiang",0000-0002-1946-2067,"[""Shanghai University""]",1.218000
8,"Hofmann, Marc",0009-0000-0193-7493,"[""Rheinland-Pfälzische Technische Universität ...",0.224333
9,"Hofmann, Marc",0009-0000-0193-7493,"[""Rheinland-Pfälzische Technische Universität ...",0.224333


In [ ]:
con = duckdb.connect(dataset_reports_db)
query = """
        SELECT*
        FROM creators_table 
        WHERE 
            primary_identifier = 0000-0003-0770-6869
        LIMIT 10
        """
display(con.execute(query).df())
con.close()

In [13]:
con = duckdb.connect(dataset_reports_db)
query = "SELECT COUNT(*) FROM creators_table WHERE creator_name IS NULL OR creator_name = ''"
result = con.execute(query).fetchone()[0]
print(result)
con.close()

223376


### Create S-index table by matching identifiers only

In [78]:
create_s_index_identifier_table(dataset_reports_db, limit=1000)

Initializing S_index_identifier (Master Researcher Profile)...
--------------------------------------------------
Success! S_index_identifier created.
Total Unique Researchers based on identifier: 27
Execution Time: 0.58 seconds
--------------------------------------------------

Top 5 Researchers (with Topic/Subfield IDs):
    primary_identifier                                primary_topic_name  \
0  0009-0001-6783-2245                       Virology and Viral Diseases   
1  0000-0002-9470-6375                 Geochemistry and Geologic Mapping   
2  0000-0002-1946-2067                    Pain Mechanisms and Treatments   
3            01es9dw61  Geological and Tectonic Studies in Latin America   
4  0000-0001-7089-7421       American Environmental and Regional History   

                    primary_subfield_name  n_datasets  S_index_topics  
0                            Epidemiology           2        2.179500  
1                 Artificial Intelligence           3        1.538500  
2

In [79]:
create_s_index_identifier_table(dataset_reports_db)

Initializing S_index_identifier (Master Researcher Profile)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--------------------------------------------------
Success! S_index_identifier created.
Total Unique Researchers based on identifier: 441,944
Execution Time: 358.95 seconds
--------------------------------------------------

Top 5 Researchers (with Topic/Subfield IDs):
                           primary_identifier  \
0                         0000-0001-5473-2109   
1  https://nrid.nii.ac.jp/nrid/1000050260047/   
2                   https://ror.org/0566bfb96   
3                         0000-0002-9160-682x   
4  https://nrid.nii.ac.jp/nrid/1000040300727/   

                     primary_topic_name            primary_subfield_name  \
0     Geochemistry and Geologic Mapping          Artificial Intelligence   
1  Magnetic confinement fusion research  Nuclear and High Energy Physics   
2     Geochemistry and Geologic Mapping          Artificial Intelligence   
3    Prenatal Screening and Diagnostics                Molecular Biology   
4  Magnetic confinement fusion research  Nuclear and Hi

In [60]:
con = duckdb.connect(dataset_reports_db)
schema_df = con.execute("DESCRIBE S_index_identifier").df()
display(schema_df)
con.close()

,column_name,column_type,null,key,default,extra
0,primary_identifier,VARCHAR,YES,None,None,None
1,creator_names,VARCHAR[],YES,None,None,None
2,name_type,VARCHAR,YES,None,None,None
3,all_affiliations,VARCHAR[],YES,None,None,None
4,primary_topic_name,VARCHAR,YES,None,None,None
5,n_unique_topics,BIGINT,YES,None,None,None
6,n_unique_subfields,BIGINT,YES,None,None,None
7,first_pub_year,INTEGER,YES,None,None,None
8,last_pub_year,INTEGER,YES,None,None,None
9,n_datasets,BIGINT,YES,None,None,None


In [113]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM S_index_identifier LIMIT 5").df())
con.close()

,primary_identifier,creator_names,name_type,all_affiliations,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,n_unique_topics,n_unique_subfields,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,0000-0001-5473-2109,"[TOKUZAWA, Tokihiko]",Personal,"[National Institute for Fusion Science, Nation...",T12157,Geochemistry and Geologic Mapping,1702,Artificial Intelligence,4499,252,...,5727142,1.393599e+06,1.400136e+06,0.243332,0.244474,0.00,0.00,0.0,0.0,14.673781
1,https://nrid.nii.ac.jp/nrid/1000050260047/,"[TANAKA, Kenji]",Personal,"[National Institute for Fusion Science, Nation...",T10346,Magnetic confinement fusion research,3106,Nuclear and High Energy Physics,4491,252,...,3208893,7.884570e+05,7.903823e+05,0.245710,0.246310,0.00,0.00,0.0,0.0,14.785334
2,https://ror.org/0566bfb96,"[Naturalis Biodiversity Center, Distributed Sy...",Organizational,[],T12157,Geochemistry and Geologic Mapping,1702,Artificial Intelligence,4255,251,...,2388894,6.805641e+05,6.805869e+05,0.284887,0.284896,1.13,1.13,1.0,1.0,17.094411
3,0000-0002-9160-682x,"[GOTO, Motoshi]",Personal,"[National Institute for Fusion Science (NIFS),...",T10978,Prenatal Screening and Diagnostics,1312,Molecular Biology,4504,252,...,2608151,6.344953e+05,6.364951e+05,0.243274,0.244041,0.00,0.00,0.0,0.0,14.659584
4,https://nrid.nii.ac.jp/nrid/1000040300727/,"[FUNABA, Hisamichi]",Personal,"[National Institute for Fusion Science (NIFS),...",T10346,Magnetic confinement fusion research,3106,Nuclear and High Energy Physics,4490,252,...,2104401,6.296659e+05,6.357325e+05,0.299214,0.302097,276241.20,0.00,197006.0,0.0,15.511266


In [112]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM S_index_identifier WHERE primary_identifier = '0000-0003-0770-6869'").df())
con.close()

,primary_identifier,creator_names,name_type,all_affiliations,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,n_unique_topics,n_unique_subfields,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,0000-0003-0770-6869,"[Gonzalez-Franco, Joan, Gonzalez-Franco, Joan D.]",None,[CUJAE],T14139,E-commerce and Technology Innovations,1403,Business and International Management,1,1,...,4,5.0,5.0,1.25,1.25,0.0,0.0,0.0,0.0,75.0


In [10]:
con = duckdb.connect(dataset_reports_db)
schema_df = con.execute("DESCRIBE S_index_identifier").df()
display(schema_df)
con.close()

,column_name,column_type,null,key,default,extra
0,primary_identifier,VARCHAR,YES,None,None,None
1,creator_names,VARCHAR[],YES,None,None,None
2,name_type,VARCHAR,YES,None,None,None
3,all_affiliations,VARCHAR[],YES,None,None,None
4,primary_topic_id,VARCHAR,YES,None,None,None
5,primary_topic_name,VARCHAR,YES,None,None,None
6,primary_subfield_id,VARCHAR,YES,None,None,None
7,primary_subfield_name,VARCHAR,YES,None,None,None
8,n_unique_topics,BIGINT,YES,None,None,None
9,n_unique_subfields,BIGINT,YES,None,None,None


### Create S-index table by matching name and affiliations set

In [5]:
create_s_index_name_affiliation_table(dataset_reports_db)

Creating s_index_name_affiliation table


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! 's_index_name_affiliation' created.
Total Unique Name-Affiliation Sets: 3,962,943
Execution Time: 289.25 seconds

Preview Top 5 Rows:
          grouping_name affiliation_set_signature  n_datasets  S_index_topics
0    nilsson, r. henrik                      None     3201095    1.619952e+06
1      abarenkov, kessy                      None     3201073    1.619924e+06
2        kõljalg, urmas                      None     3201065    1.619920e+06
3  larsson, karl-henrik                      None     3200948    1.619337e+06
4        tedersoo, leho                      None     2383786    1.224757e+06


In [6]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM S_index_name_affiliation LIMIT 5").df())
con.close()

,grouping_name,affiliation_set_signature,name_type,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,n_unique_topics,n_unique_subfields,first_pub_year,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,"nilsson, r. henrik",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2012,...,3201095,1.619952e+06,1.627984e+06,0.506062,0.508571,864.65,814.86,625.0,590.0,31.049449
1,"abarenkov, kessy",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2012,...,3201073,1.619924e+06,1.627956e+06,0.506056,0.508566,862.65,812.86,623.0,588.0,31.049163
2,"kõljalg, urmas",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2015,...,3201065,1.619920e+06,1.627953e+06,0.506057,0.508566,864.87,815.08,625.0,590.0,31.049142
3,"larsson, karl-henrik",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2015,...,3200948,1.619337e+06,1.627369e+06,0.505893,0.508402,57.90,57.90,43.0,43.0,31.049087
4,"tedersoo, leho",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4471,252,2015,...,2383786,1.224757e+06,1.228910e+06,0.513787,0.515529,2009.15,69.15,1992.0,53.0,30.919935


In [9]:
con = duckdb.connect(dataset_reports_db)
result = con.execute("SELECT SUM(n_datasets) FROM S_index_name_affiliation").fetchone()[0]
print(result)
con.close()

216465136


In [17]:
con = duckdb.connect(dataset_reports_db)
schema_df = con.execute("DESCRIBE S_index_name_affiliation").df()
display(schema_df)
con.close()

,column_name,column_type,null,key,default,extra
0,grouping_name,VARCHAR,YES,None,None,None
1,affiliation_set_signature,VARCHAR,YES,None,None,None
2,name_type,VARCHAR,YES,None,None,None
3,primary_topic_id,VARCHAR,YES,None,None,None
4,primary_topic_name,VARCHAR,YES,None,None,None
5,primary_subfield_id,VARCHAR,YES,None,None,None
6,primary_subfield_name,VARCHAR,YES,None,None,None
7,n_unique_topics,BIGINT,YES,None,None,None
8,n_unique_subfields,BIGINT,YES,None,None,None
9,first_pub_year,INTEGER,YES,None,None,None


### Create S-index table by matching name

In [ ]:
create_s_index_name_table(dataset_reports_db)

Initializing S_index_name table


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM S_index_name LIMIT 5                     ").df())
con.close()

,grouping_name,name_variations,n_unique_affiliations,n_associated_identifiers,name_type,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,n_unique_topics,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,"narushima, yoshiro","[NARUSHIMA, Yoshiro]",1,1,Personal,T10255,Oceanographic and Atmospheric Processes,1312,Molecular Biology,343,...,1304,329.683863,337.202989,0.252825,0.258591,0.0,0.0,0.0,0.0,15.524571
1,"funaba, hisamichi","[FUNABA, Hisamichi]",1,1,Personal,NaN,NaN,NaN,NaN,0,...,694,185.332333,185.332333,0.267049,0.267049,0.0,0.0,0.0,0.0,16.022968
2,"toraman, gözdenur","[Toraman, Gözdenur]",2,1,Personal,T11213,Genomic variations and chromosomal abnormalities,1311,Genetics,1,...,1,1.346167,1.346167,1.346167,1.346167,0.0,0.0,0.0,0.0,80.770000
3,gbif.org user,[GBIF.org User],0,0,Organizational,NaN,NaN,NaN,NaN,0,...,1,0.224333,0.224333,0.224333,0.224333,0.0,0.0,0.0,0.0,13.460000
